# Build Dataset: MSLP Regression Cache

Build the mean sea level pressure DJF regression cache against Niño3.4 anomaly.

This notebook keeps the exact DJF regression logic from `regression_mc.ipynb`, but strips all plotting. The output is a NetCDF cache that can be reused later when plot styling or annotations change.

# 1. Import Library

In [ ]:

from pathlib import Path
import numpy as np
import pandas as pd
import xarray as xr
from scipy.stats import norm, t as student_t

# Keep the same study window and split years as regression_mc.ipynb.
FULL_YEARS = np.arange(1981, 2026)
PAST_YEARS = np.arange(1981, 2007)
RECENT_YEARS = np.arange(2007, 2026)
ANALYSIS_START = '1980-12-01'
ANALYSIS_END = '2025-02-28'
MC_LAT = slice(32, -32)
MC_LON = slice(58, 272)
GRAVITY = 9.80665
EARTH_RADIUS = 6371000.0

RAINFALL_PATH = Path('/Users/rizzie/ClimateData/mswep-monthly/mswep_monthly_combined.nc').resolve()
WIND_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/uv_850_global_1979-2025.nc').resolve()
MSLP_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/mslp_global_1979-2025.nc').resolve()
GEO850_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/geopotensial_850_global_1979-2025.nc').resolve()
MFC_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/viwve_viwvn_global_1979-2025.nc').resolve()
SVP_PATH = Path('/Users/rizzie/ClimateData/era5-monthly/psi_chi_850_global_1979-2025.nc').resolve()
NINO34_PATH = Path('/Users/rizzie/ClimateData/index-monthly/nino34.anom.csv').resolve()
NINO34_COLUMN = '   Nino Anom 3.4 Index  using ersstv5 from CPC  missing value -99.99 https://psl.noaa.gov/data/timeseries/month/'


def standardize_obj(obj):
    rename_map = {}
    if 'valid_time' in obj.coords or 'valid_time' in obj.dims:
        rename_map['valid_time'] = 'time'
    if 'latitude' in obj.coords or 'latitude' in obj.dims:
        rename_map['latitude'] = 'lat'
    if 'longitude' in obj.coords or 'longitude' in obj.dims:
        rename_map['longitude'] = 'lon'
    if rename_map:
        obj = obj.rename(rename_map)

    if 'lon' in obj.coords:
        obj = obj.assign_coords(lon=(obj.lon % 360)).sortby('lon')

    if 'lat' in obj.coords:
        lat_values = np.asarray(obj['lat'].values)
        if lat_values.size > 1 and lat_values[0] < lat_values[-1]:
            obj = obj.sortby('lat', ascending=False)

    return obj


def select_850_hpa_if_present(obj):
    for level_name in ('pressure_level', 'level', 'lev'):
        if level_name not in obj.coords and level_name not in obj.dims:
            continue

        level_values = np.atleast_1d(np.asarray(obj[level_name].values))
        for target in (850, 850.0, 85000, 85000.0):
            if target in level_values:
                return obj.sel({level_name: target})

        raise ValueError(
            f"850 hPa level not found in '{level_name}'. Available values include: "
            f"{level_values[:10]}"
        )

    return obj


def match_variable_name(ds, target):
    target_key = target.replace('_', '').lower()
    matches = [
        name
        for name in ds.data_vars
        if name.lower() == target.lower()
        or name.lower().replace('_', '') == target_key
    ]
    if not matches:
        matches = [
            name
            for name in ds.data_vars
            if target_key in name.lower().replace('_', '')
        ]
    if not matches:
        raise KeyError(f'Variable for {target} not found in the dataset')
    return matches[0]


def crop_analysis_domain(obj):
    if 'lat' in obj.coords or 'lat' in obj.dims:
        obj = obj.sel(lat=MC_LAT)
    if 'lon' in obj.coords or 'lon' in obj.dims:
        obj = obj.sel(lon=MC_LON)
    return obj


def build_djf_seasonal_means(da, start=ANALYSIS_START, end=ANALYSIS_END):
    da = da.sel(time=slice(start, end))
    da = crop_analysis_domain(da)
    month_mask = da.time.dt.month.isin((12, 1, 2))
    djf_year = xr.where(da.time.dt.month == 12, da.time.dt.year + 1, da.time.dt.year)
    da = da.sel(time=month_mask).assign_coords(
        djf_year=('time', djf_year.sel(time=month_mask).data)
    )
    da = da.sel(time=da.djf_year.isin(FULL_YEARS))
    clim = da
    past = clim.sel(time=clim.djf_year.isin(PAST_YEARS))
    recent = clim.sel(time=clim.djf_year.isin(RECENT_YEARS))
    return (
        clim.groupby('djf_year').mean('time'),
        past.groupby('djf_year').mean('time'),
        recent.groupby('djf_year').mean('time'),
    )


def build_nino34_djf_series(df):
    df = df.set_index('Date').loc['1980-12-01':'2025-02-28']
    df = df[df.index.month.isin((12, 1, 2))].copy()
    df['djf_year'] = df.index.year + (df.index.month == 12).astype('int8')
    df = df[df['djf_year'].isin(FULL_YEARS)].copy()

    clim_series = df.groupby('djf_year')[NINO34_COLUMN].mean()
    past_series = df[df['djf_year'].isin(PAST_YEARS)].groupby('djf_year')[NINO34_COLUMN].mean()
    recent_series = df[df['djf_year'].isin(RECENT_YEARS)].groupby('djf_year')[NINO34_COLUMN].mean()

    clim = xr.DataArray(
        clim_series.to_numpy(),
        coords={'djf_year': clim_series.index.to_numpy()},
        dims='djf_year',
        name='nino34',
    )
    past = xr.DataArray(
        past_series.to_numpy(),
        coords={'djf_year': past_series.index.to_numpy()},
        dims='djf_year',
        name='nino34',
    )
    recent = xr.DataArray(
        recent_series.to_numpy(),
        coords={'djf_year': recent_series.index.to_numpy()},
        dims='djf_year',
        name='nino34',
    )
    return clim, past, recent


def safe_corr(corr):
    return xr.where(np.abs(corr) >= 0.999999, np.sign(corr) * 0.999999, corr)




def _finite_mask(*arrays):
    mask = None
    for arr in arrays:
        current = xr.apply_ufunc(np.isfinite, arr)
        mask = current if mask is None else (mask & current)
    return mask


def simple_regression_stats(field, index, sample_dim):
    field, index = xr.align(field, index, join='inner')
    valid = _finite_mask(field, index)
    x = index.where(valid)
    y = field.where(valid)

    n = valid.sum(sample_dim).astype(np.float32)
    safe_n = xr.where(n > 0, n, np.nan)

    sx = x.sum(sample_dim, skipna=True)
    sy = y.sum(sample_dim, skipna=True)
    sxx = (x * x).sum(sample_dim, skipna=True)
    syy = (y * y).sum(sample_dim, skipna=True)
    sxy = (x * y).sum(sample_dim, skipna=True)

    sxx_c = sxx - (sx ** 2 / safe_n)
    sxy_c = sxy - (sx * sy / safe_n)
    syy_c = syy - (sy ** 2 / safe_n)

    slope = sxy_c / sxx_c
    df = safe_n - 2
    sse = syy_c - (sxy_c ** 2 / sxx_c)
    mse = sse / df
    se = np.sqrt(mse / sxx_c)
    t_stat = slope / se
    p = xr.apply_ufunc(
        lambda t, dof: 2 * student_t.sf(np.abs(t), df=dof),
        t_stat,
        df,
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float],
    )

    valid_stats = (n >= 3) & np.isfinite(slope) & np.isfinite(p)
    slope = slope.where(valid_stats)
    p = p.where(valid_stats)
    return slope.astype(np.float32), p.astype(np.float32), n


def pooled_ols_interaction_stats(field_past, field_recent, index_past, index_recent, sample_dim):
    field_past, index_past = xr.align(field_past, index_past, join='inner')
    field_recent, index_recent = xr.align(field_recent, index_recent, join='inner')

    valid_past = _finite_mask(field_past, index_past)
    valid_recent = _finite_mask(field_recent, index_recent)

    x0 = index_past.where(valid_past)
    y0 = field_past.where(valid_past)
    x1 = index_recent.where(valid_recent)
    y1 = field_recent.where(valid_recent)

    n0 = valid_past.sum(sample_dim).astype(np.float32)
    n1 = valid_recent.sum(sample_dim).astype(np.float32)
    safe_n0 = xr.where(n0 > 0, n0, np.nan)
    safe_n1 = xr.where(n1 > 0, n1, np.nan)

    sx0 = x0.sum(sample_dim, skipna=True)
    sy0 = y0.sum(sample_dim, skipna=True)
    sxx0 = (x0 * x0).sum(sample_dim, skipna=True)
    syy0 = (y0 * y0).sum(sample_dim, skipna=True)
    sxy0 = (x0 * y0).sum(sample_dim, skipna=True)

    sx1 = x1.sum(sample_dim, skipna=True)
    sy1 = y1.sum(sample_dim, skipna=True)
    sxx1 = (x1 * x1).sum(sample_dim, skipna=True)
    syy1 = (y1 * y1).sum(sample_dim, skipna=True)
    sxy1 = (x1 * y1).sum(sample_dim, skipna=True)

    beta_names = ['b0', 'b1', 'b2', 'b3']
    xtx = xr.concat(
        [
            xr.concat([safe_n0 + safe_n1, sx0 + sx1, safe_n1, sx1], dim='col'),
            xr.concat([sx0 + sx1, sxx0 + sxx1, sx1, sxx1], dim='col'),
            xr.concat([safe_n1, sx1, safe_n1, sx1], dim='col'),
            xr.concat([sx1, sxx1, sx1, sxx1], dim='col'),
        ],
        dim='row',
    ).assign_coords(row=beta_names, col=beta_names)
    xty = xr.concat([sy0 + sy1, sxy0 + sxy1, sy1, sxy1], dim='col').assign_coords(col=beta_names)

    xtx_pinv = xr.apply_ufunc(
        np.linalg.pinv,
        xtx,
        input_core_dims=[['row', 'col']],
        output_core_dims=[['row', 'col']],
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float],
    )
    beta = xr.dot(xtx_pinv, xty, dims='col')

    yty = syy0 + syy1
    beta_dot_xty = xr.dot(beta, xty, dims='col')
    rss = yty - beta_dot_xty
    df = safe_n0 + safe_n1 - 4
    sigma2 = rss / df
    se_diff = np.sqrt(sigma2 * xtx_pinv.sel(row='b3', col='b3'))
    diff = beta.sel(row='b3')
    t_stat = diff / se_diff
    p = xr.apply_ufunc(
        lambda t, dof: 2 * student_t.sf(np.abs(t), df=dof),
        t_stat,
        df,
        vectorize=True,
        dask='parallelized',
        output_dtypes=[float],
    )

    valid_stats = (df > 0) & np.isfinite(diff) & np.isfinite(p)
    diff = diff.where(valid_stats)
    p = p.where(valid_stats)
    return diff.astype(np.float32), p.astype(np.float32), df.astype(np.float32)


def reg_and_pvalues(field_clim, field_past, field_recent, nino_clim, nino_past, nino_recent):
    reg_clim, p_clim, _ = simple_regression_stats(field_clim, nino_clim, 'djf_year')
    reg_past, p_past, _ = simple_regression_stats(field_past, nino_past, 'djf_year')
    reg_recent, p_recent, _ = simple_regression_stats(field_recent, nino_recent, 'djf_year')
    reg_recent_minus_past, p_recent_minus_past, _ = pooled_ols_interaction_stats(
        field_past,
        field_recent,
        nino_past,
        nino_recent,
        'djf_year',
    )

    return {
        'reg_clim': reg_clim,
        'reg_past': reg_past,
        'reg_recent': reg_recent,
        'reg_recent_minus_past': reg_recent_minus_past,
        'p_clim': p_clim,
        'p_past': p_past,
        'p_recent': p_recent,
        'p_recent_minus_past': p_recent_minus_past,
    }
def cast_float_and_mask_vars(ds):
    ds = ds.copy()
    for name in list(ds.data_vars):
        dtype = ds[name].dtype
        if np.issubdtype(dtype, np.floating):
            ds[name] = ds[name].astype('float32')
        elif np.issubdtype(dtype, np.bool_):
            ds[name] = ds[name].astype('int8')
    return ds


def build_encoding(ds):
    return {name: {'zlib': True, 'complevel': 4} for name in ds.data_vars}


def write_dataset(ds, path, label):
    ds = cast_float_and_mask_vars(ds)
    print(f'Writing {label}: {path}')
    ds.to_netcdf(path, encoding=build_encoding(ds))


# 2. Open Data

In [ ]:
mslp_path = MSLP_PATH
mslp_chunks = {'valid_time': 12}

ds_mslp = xr.open_dataset(mslp_path, chunks=mslp_chunks)['msl']
ds_mslp = standardize_obj(ds_mslp)
mslp_raw = ds_mslp

# 3. Regression & Significance

In [ ]:
df_nino34 = pd.read_csv(
    NINO34_PATH,
    usecols=['Date', NINO34_COLUMN],
    parse_dates=['Date'],
)
nino34_clim, nino34_past, nino34_recent = build_nino34_djf_series(df_nino34)

mslp_clim, mslp_past, mslp_recent = build_djf_seasonal_means(mslp_raw)
mslp_stats = reg_and_pvalues(
    mslp_clim, mslp_past, mslp_recent,
    nino34_clim, nino34_past, nino34_recent,
)

# 4. Save Cache

In [ ]:
cache_path = Path('/Users/rizzie/TugasAkhir/data_processing/data/intermediate/divided_regression/mslp_reg_cache_1981_2025.nc').resolve()
cache_path.parent.mkdir(parents=True, exist_ok=True)

cache_ds = xr.Dataset(
    {
        'mslp_reg_clim': mslp_stats['reg_clim'],
        'mslp_reg_past': mslp_stats['reg_past'],
        'mslp_reg_recent': mslp_stats['reg_recent'],
        'mslp_reg_recent_minus_past': mslp_stats['reg_recent_minus_past'],
        'mslp_reg_clim_p': mslp_stats['p_clim'],
        'mslp_reg_past_p': mslp_stats['p_past'],
        'mslp_reg_recent_p': mslp_stats['p_recent'],
        'mslp_reg_recent_minus_past_p': mslp_stats['p_recent_minus_past'],
        'mslp_reg_clim_sig': (mslp_stats['p_clim'] < 0.05),
        'mslp_reg_past_sig': (mslp_stats['p_past'] < 0.05),
        'mslp_reg_recent_sig': (mslp_stats['p_recent'] < 0.05),
        'mslp_reg_recent_minus_past_sig': (mslp_stats['p_recent_minus_past'] < 0.05),
    }
)
cache_ds.attrs.update(
    {
        'title': 'MSLP regression cache',
        'period': '1981-2025',
        'split_p1': '1981-2006',
        'split_p2': '2007-2025',
        'note': 'MSLP uses raw monthly values; Niño3.4 stays on the anomaly CSV.',
    }
)
write_dataset(cache_ds, cache_path, 'mslp cache')